# IMPORTACIONES Y CSVS

In [2]:
import pandas as pd

enzimas = pd.read_csv("enzimas.csv")
excepciones = pd.read_csv("excepciones.csv")
sidder_atc = pd.read_csv("drug_atc.tsv", sep="\t", header=None, names=["flat", "drug_atc"])
sidder = pd.read_csv("sidder_eadv.csv")
codigos_ATC = pd.read_csv("ATC_DB.csv")

# Primero trato el csv de enzimas

In [4]:
#Como se hizo en el notebook de ejecucion de las bases de datos:

# Diccionario que cuenta cuantas encimas tengo de cada y cuales son el top 5 con las que me quedare
diccionario_enzimas = (
    enzimas[enzimas["type"] == "enzyme"]["gene_name"]
    .value_counts()
    .to_dict()
)

print(diccionario_enzimas)
# Por tanto escogemos las que se metabolizan por cualquiera de las siguientes enzimas ["CYP2D6", "CYP3A4", "CYP3A5", "CYP2C19", "CYP2C9", "CYP1A2"] 
#que son las top 5 y CYP3A5 porque va con CYP3A4 (explicar mejor) y las target porque aunque no den problemas por las interacciones son posibles 
#opciones de reemplazo de principio activo
target_top5 = enzimas[
    ((enzimas["type"] == "enzyme") & (enzimas["gene_name"].isin(["CYP2D6", "CYP3A4", "CYP3A5", "CYP2C19", "CYP2C9", "CYP1A2"])))
    |       #or
    (enzimas["type"] == "target")
]
#Elimino la columna de sinonimos y de uniprot_id
target_top5 = target_top5.drop(columns=["synonyms", "uniprot_id"])
# Renombro las columnas 
target_top5.columns = ["DrugBank_id", "Drug_name", "Tipo", "Gene_name", "Accion"]
target_top5.head()

{'CYP3A4': 1119, 'CYP2C9': 435, 'CYP2D6': 432, 'CYP1A2': 385, 'CYP2C19': 350, 'CYP2C8': 336, 'CYP3A5': 297, 'CYP2B6': 232, 'UGT1A1': 148, 'CYP2E1': 144, 'CYP3A7': 128, 'CYP1A1': 112, 'UGT1A9': 86, 'CYP2A6': 85, 'UGT2B7': 84, 'BCHE': 61, 'UGT1A3': 58, 'CYP1B1': 45, 'CYP19A1': 42, 'MAOA': 40, 'UGT1A4': 32, 'CES1': 32, 'PTGS1': 31, 'PTGS2': 29, 'CYP2C18': 29, 'MAOB': 24, 'UGT1A8': 23, 'CYP2J2': 23, 'MPO': 22, 'XDH': 22, 'AOX1': 21, 'UGT2B4': 20, 'FMO3': 19, 'UGT2B15': 19, 'UGT1A10': 18, 'UGT1A6': 18, 'CYP11B1': 17, 'COMT': 16, 'CYP3A43': 15, 'SULT1A1': 14, 'CYP4A11': 13, 'CBR1': 13, 'CYP11B2': 13, 'UGT2B17': 12, 'UGT1A7': 12, 'SOD1': 12, 'GSTP1': 12, 'DCK': 12, 'GSTA1': 11, 'FMO1': 11, 'POR': 11, 'HSD11B1': 11, 'GSTM1': 10, 'CYP11A1': 10, 'NAT2': 10, 'CYP4F2': 10, 'NME1': 10, 'IDE': 10, 'CES2': 10, 'PLA2G4A': 9, 'SULT1A3': 8, 'ADA': 8, 'GSTA2': 8, 'AMY2A': 8, 'DDC': 8, 'CYP17A1': 8, 'NQO1': 8, 'ACHE': 8, 'GLUL': 7, 'CYP2A13': 7, 'MTHFR': 7, 'TYMP': 7, 'UGT2B10': 7, 'ALOX5': 7, 'ALDH2': 7,

,DrugBank_id,Drug_name,Tipo,Gene_name,Accion
0,DB00001,Lepirudin,target,F2,inhibitor
1,DB00002,Cetuximab,target,EGFR,binder
2,DB00002,Cetuximab,target,FCGR3B,binder
3,DB00002,Cetuximab,target,C1QA,binder
4,DB00002,Cetuximab,target,C1QB,binder


In [5]:
# VER SI HAY NULOS
target_top5.isna().any()

DrugBank_id    False
Drug_name      False
Tipo           False
Gene_name       True
Accion          True
dtype: bool

In [6]:
#Hay nulos en accion cuando solo hay enzimas y no targets, por tanto se tendra en cuenta
comprobacion = target_top5[target_top5["Tipo"]=="enzyme"]
print(comprobacion.head())
comprobacion.isna().any()

   DrugBank_id              Drug_name    Tipo Gene_name     Accion
28     DB00008  Peginterferon alfa-2a  enzyme    CYP1A2  inhibitor
36     DB00011     Interferon alfa-n1  enzyme    CYP1A2  inhibitor
58     DB00018     Interferon alfa-n3  enzyme    CYP1A2  inhibitor
68     DB00022  Peginterferon alfa-2b  enzyme    CYP1A2  inhibitor
69     DB00022  Peginterferon alfa-2b  enzyme    CYP2D6  inhibitor


DrugBank_id    False
Drug_name      False
Tipo           False
Gene_name      False
Accion          True
dtype: bool

### TRATO LOS CODIGOS ATC QUE OBTUVIMOS DESPUES

In [8]:
# Eliminamos la columna que corresponde a el nombre para no repetir cuando unamos
codigos_ATC = codigos_ATC.drop(columns=["drug_name"])
# Elimino los que como codigo ATC aparezca Nan para separarlos
codigos_ATC = codigos_ATC.dropna(subset=['atc_codes'])

#Como un drug name puede tener mas de un codigo ATC los separo en diferentes filas
separado = codigos_ATC["atc_codes"].str.split(r"\|")
#Donde almaceno las filas
rows = []
#identificador de la fila donde me llego
i = 0
#Por cada codigo
for cods in separado:
    fila = codigos_ATC.iloc[i]
    if len(cods)!=1:
        d_id = fila["drugbank_id"]
        for uno in cods:
            rows.append([d_id, uno])
    else:
        rows.append(fila.tolist())
    i+=1

#Cambio los nombres para estandarizarlo
ATC_separado = pd.DataFrame(rows, columns=[
        "DrugBank_id",
        "Drug_ATC"
    ])
ATC_separado.head()

,DrugBank_id,Drug_ATC
0,DB00001,B01AE02
1,DB00002,L01FE01
2,DB00003,R05CB13
3,DB00004,L01XX29
4,DB00005,L04AB01


In [9]:
# Ver si hay nulos
ATC_separado.isna().any()

DrugBank_id    False
Drug_ATC       False
dtype: bool

### Uno ambos DF para crear el de drugbank

In [11]:
#Por la derecha porque quiero que se me queden todos los que tiene DB
drugbank = pd.merge (ATC_separado, target_top5, on="DrugBank_id", how="right")
print(drugbank.shape)
drugbank.head()

(38466, 6)


,DrugBank_id,Drug_ATC,Drug_name,Tipo,Gene_name,Accion
0,DB00001,B01AE02,Lepirudin,target,F2,inhibitor
1,DB00002,L01FE01,Cetuximab,target,EGFR,binder
2,DB00002,L01FE01,Cetuximab,target,FCGR3B,binder
3,DB00002,L01FE01,Cetuximab,target,C1QA,binder
4,DB00002,L01FE01,Cetuximab,target,C1QB,binder


### Añado la columna: Prioridad

In [13]:
# Del df de enzimas nos quedamos solo con las top 5
top_5 = enzimas[enzimas["gene_name"].isin(["CYP2D6", "CYP3A4", "CYP3A5", "CYP2C19", "CYP2C9", "CYP1A2"])]
# Y de esas solo con las duplicadas que significa que tienen mas de una enzima
duplicados = top_5[top_5.duplicated(subset="drugbank_id", keep=False)]
print(len(duplicados["drugbank_id"].unique()))

#De las que hemos querido establecer prioridad son:
problematicas = duplicados["drug_name"].unique().tolist()

#Añadimos la columna prioridad
#Para todas las que se metabolizan por una enzima (las que no son problemáticas) ponemos 1 
#En este caso tambien se le pone a los target pero como no los usamos luego, es indiferente
drugbank["Prioridad"] = drugbank["Drug_name"].apply(
    lambda x: 1 if x not in problematicas else None
)
#Lo junto con las excepciones que hemos podido obtener
DDI_prioridad = drugbank.merge(excepciones, how="left", on=["Drug_name", "Gene_name"])
DDI_prioridad.head()

755


,DrugBank_id,Drug_ATC,Drug_name,Tipo,Gene_name,Accion,Prioridad_x,Prioridad_y
0,DB00001,B01AE02,Lepirudin,target,F2,inhibitor,1.0,NaN
1,DB00002,L01FE01,Cetuximab,target,EGFR,binder,1.0,NaN
2,DB00002,L01FE01,Cetuximab,target,FCGR3B,binder,1.0,NaN
3,DB00002,L01FE01,Cetuximab,target,C1QA,binder,1.0,NaN
4,DB00002,L01FE01,Cetuximab,target,C1QB,binder,1.0,NaN


In [14]:
# Unimos x e y
DDI_prioridad["Prioridad"] = DDI_prioridad["Prioridad_x"].fillna(DDI_prioridad["Prioridad_y"])
DDI_prioridad= DDI_prioridad.drop(columns=["Prioridad_x", "Prioridad_y"])
DDI_prioridad.head()
#Rellenamos los que aun son nulos con 0 porque:
# son de los que: o no hemos podido obtener prioridad, o habia otra enzima que notebookLM identifico como principal de las que no tengo en cuenta
DDI_prioridad["Prioridad"] = DDI_prioridad["Prioridad"].fillna(0)
#Porsiacaso
DDI_prioridad = DDI_prioridad.drop_duplicates()
DDI_prioridad.isna().any()

DrugBank_id    False
Drug_ATC        True
Drug_name      False
Tipo           False
Gene_name       True
Accion          True
Prioridad      False
dtype: bool

In [15]:
#LO GUARDO
DDI_prioridad.to_csv("DDI_sea.csv", index=False)

# UNO AL DF DE SIDDER LOS CODIGOS ATC 
Es para poder correlacionar csvs de alguna manera

In [17]:
efectos_adversos = pd.merge (sidder_atc, sidder, on="flat", how="inner")
# Sin nulos
efectos_adversos.isna().any()

flat              False
drug_atc          False
drug_name         False
side_effect_se    False
freq_mean_y       False
dtype: bool

In [18]:
#liminamos la columna de flat que no usaremos
efectos_adversos = efectos_adversos.drop(columns=["flat"])
#Cambio los nombres de las columnas para estandarizar
efectos_adversos.columns = ["Drug_ATC","Drug_name","Side_effect","Freq_media"]
efectos_adversos.head()

,Drug_ATC,Drug_name,Side_effect,Freq_media
0,A16AA01,carnitine,Abdominal pain,0.116000
1,A16AA01,carnitine,Gastrointestinal pain,0.116000
2,A16AA01,carnitine,Amblyopia,0.036667
3,A16AA01,carnitine,Anaemia,0.058000
4,A16AA01,carnitine,Decreased appetite,0.042500


In [19]:
#Lo guardo
efectos_adversos.to_csv("efectos_adversos.csv", index=False)